# Project Title:  TexasSalaryPrediction

**Project Description**: This database has salary information for positions at all **113 agencies** in the Texas state government. 

---

| Field | Details |
|---|---|
| **Project Title** | Texas Salary Prediction |
| **Project Team ID** | PTID-CDS-MAY-26-11194 |
| **Project ID** | PRCP-1024 |
| **Institute** | Datamites |
| **Domain** | Public Sector / Government HR Analytics |
| **Target Variable** | `ANNUAL` (Annual Salary) |
| **Problem Type** | Supervised Machine Learning — Regression |

---


## 1. Define the Goal:

**Task 1**:- Prepare a complete **data analysis report** on the given data of `Texas Salary data`.

**Task 2**:- Create **`a predictive model`** which will help the **Texas state government  team** to know the `payroll information` of **employees** of **`the state of Texas`**.  

**Task 3**:-
1. Who are the `outliers` in the salaries?
2. What `departments`/`roles` have the biggest `wage`/`salary disparities` between `managers` and `employees`?
3. Have salaries and total compensations for some `roles`/ `departments`/ head-count changed over time?


In [ ]:
import pandas as pd
import numpy as np
import warnings
import pickle
from datetime import datetime

# Visualization
import seaborn as sns
import matplotlib.pyplot as plt

# Preprocessing
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer

# Model Selection
from sklearn.model_selection import train_test_split, RandomizedSearchCV, cross_val_score

# Models
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

# Evaluation
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Pipeline
from sklearn.pipeline import Pipeline

# Settings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

print("✅ All libraries imported successfully.")

## 2. Import Dataset:

In [ ]:
df = pd.read_csv("salary.csv")

In [ ]:
pd.set_option('display.max_columns', None) 

In [ ]:
pd.set_option('display.max_colwidth', None)

In [ ]:
df.head()

## 3. Domain Analysis:

**Texas Employee Salary Domain Analysis Framework**:
  
**1. Core Business Logic & Legal Context**

 - The data contains raw identifiers or **employees details** (`FIRST NAME`, `LAST NAME`,`MI` ,`STATE NUMBER`) used for government transparency.
   
 - *The `STATUS` Column*:  You must classify workers into **Full-Time, Part-Time, Elected Official**, or **Contracted statuses**, as this dictates their salary structures.

**2. Column-Level Domain Mapping**

 - `AGENCY` & `AGENCY NAME`: Represents the specific **state department** 
 - `CLASS CODE` & `CLASS TITLE`: The official **state job classification index**. This column links lower-level staff to management.

    **Employee Demographics**

 - `ETHNICITY` & `GENDER`: Used in public sectors for **equity reporting and compliance audits**. 

    **Compensation Tracking (The Core Logic)**

- `HRLY RATE`, `HRS PER WK`, `MONTHLY`, `ANNUAL`: Redundant representations of **pay**.
  - Calculation rule: $\text{HRLY RATE} \times \text{HRS PER WK} \times 52 \text{ weeks} \approx \text{ANNUAL}$.
    
- You must verify if `ANNUAL` represents actual base **pay** or includes **overtime** and **longevity pay**.
- `summed_annual_salary`: Indicates **the total fiscal impact** per **employee** across *all jobs* held.
  
**Employment Status Flags (The Constraints)**

- `duplicated`, `multiple_full_time_jobs`, `combined_multiple_jobs`:

  Texas law allows employees to work for multiple **state agencies** under specific dual-employment provisions. These flags prevent you from double-counting employee headcounts during aggregations.

- `hide_from_search` **flag** does not describe an employee's role, skills, experience, or compensation structure.so it typically contributes little predictive information and may introduce unnecessary noise into the model.
  
- `EMPLOY DATE`: Tracks tenure.

  - Domain Knowledge:
     - **State pay scales** often increase automatically based on ``years of service`` (longevity pay)

## 4. Basic checks:

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
len(df.columns)

In [ ]:
df.info()

In [ ]:
df.describe().T

In [ ]:
df.describe(include='O').T

**Basic cheks Insights**:
- In this Datasets, **total number of columns** are **21**.

- The **numerical columns** are `AGENCY`, `HRLY RATE`, `HRS PER WK`, `MONTHLY`, `ANNUAL`, `STATE NUMBER`, `multiple_full_time_jobs`, `summed_annual_salary`.

- The **categorical columns** are `AGENCY NAME`, `LAST NAME`, `FIRST NAME`, `MI`, `CLASS CODE`, `CLASS TITLE`, `ETHNICITY`, `GENDER`, `STATUS`, `EMPLOY DATE`, `duplicated`, `combined_multiple_jobs`, `hide_from_search`.

- some categorial columns like `CLASS CODE` is actually **integer** and `EMPLOY DATE` is actually **DATE** type but they are
  store as **TEXT**.

- And columns like `duplicated`,`combined_multiple_jobs`,`combined_multiple_jobs` are bollion types.

- And **NULL values containts columns** are `duplicated`, `multiple_full_time_jobs`, `combined_multiple_jobs`,`summed_annual_salary`,`hide_from_search`.

## 5. EDA (Visualization to find Insights):

In [ ]:
df.isnull().sum()

In [ ]:
df.duplicated().sum()

In [ ]:
null_cols = df.columns[df.isnull().any()]
print(null_cols)

In [ ]:
num_cols = df.select_dtypes(include=['number']).columns.tolist()
print("Numerical columns:",num_cols)
print("-------------------------")
print("No. of numerical columns are:",len(num_cols))
print("\n")
cat_cols = df.select_dtypes(include= ['object']).columns.tolist()
print("Categorical columns:",cat_cols)
print("-------------------------")
print("No. of Categorical columns are : ", len(cat_cols))

### **Univariate Anaylysis:**

 **Histplot**

In [ ]:

pltn = 1
plt.figure(figsize=(20,10))

for col in num_cols:
  plt.subplot(3,4,pltn)
  sns.histplot(df[col], kde=True, bins=20)
  plt.title(f"{col} Distribution")
  pltn += 1
plt.suptitle('Univariate Analysis — Histograms', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

**Insights**:
- Most salary-related columns `ANNUAL` and `MONTHLY` are **right-skewed** with some **high-value outliers**.

-  `HRS PER WK` shows most employees work around **40 hours/week**.

- `multiple_full_time_jobs` has very low variation, so it may not be useful for prediction.

-`STATE NUMBER` behaves like a categorical feature, not numerical data.

- Salary distributions are not normal, so outlier handling or log transformation may help before modeling.

In [ ]:
cat_cols

**Note**:
> You should remove columns that are **Unique identifiers, Personal names, Too many unique values, Not useful for relationship analysis.**.
>  Hence, here we removed `LAST NAME`, `FIRST NAME`, `MI`, `.EMPLOY DATE` and `CLASS TITLE` ( if CLASS CODE already exists).

In [ ]:
new_cat_cols = [
 'AGENCY NAME',
 'CLASS CODE',
 'ETHNICITY',
 'GENDER',
 'STATUS',
 'duplicated',
 'combined_multiple_jobs',
 'hide_from_search'
]

In [ ]:
plt.figure(figsize=(25, 20))
for i, col in enumerate(new_cat_cols, 1): #Loop through `new_cat_cols` and provide an index starting from 1 instead of the default 0.
    plt.subplot(4, 3, i)
    top_vals = df[col].value_counts().head(10).index
    sns.countplot(x=col, data= df[df[col].isin(top_vals)], order=top_vals, palette='Set2')
    #--
    plt.title(f'Distribution of {col}', fontsize=16)
    plt.xticks(rotation=45, ha='right') 
    plt.xlabel('')
    plt.ylabel('Count')
    #--
plt.suptitle('Categorical Feature Distributions (Top 10 Values)', fontsize=15, y=1.01)
plt.tight_layout()
plt.show()

**Insights**:
* `AGENCY NAME` and `CLASS CODE` have too many categories, making the plots crowded and hard to interpret.
* Most employees belong to a few major agencies and class codes, while many categories have very low frequency.
* `ETHNICITY` shows most employees are from `WHITE`, `HISPANIC`, and `BLACK` groups.
* `GENDER` distribution indicates female employees are slightly higher than male employees.
* `STATUS` is dominated by a single employment type, showing class imbalance in employee status categories.
* `duplicated` contains only `True`, so this feature provides no useful variation for analysis or modeling.


### **Bivariate Analysis:**

In [ ]:
num_dis = df.select_dtypes(include=["int64"]).columns
num_counti = df.select_dtypes(include=["float64"]).columns
print('numeric dis columns: \n', num_dis)
print(len(num_dis))
print('numeric counti columns: \n', num_counti)
print(len(num_counti)) 

>  When **a categorical column** has *hundreds* or *thousands* of categories (like `AGENCY`, `AGENCY NAME`, `CLASS CODE`), `sns.barplot()` or `sns.countplot()` becomes very **slow** and **unreadable**.

> Show Only **Top N** Categories

In [ ]:
top_n = 15 # visualize only the top 15 categories.
plt.figure(figsize=(15, 6))

for i, col in enumerate(num_dis, 1): #the loop runs twice #i=1 for AGENCY #i=2 for STATE NUMBER

    top_categories = (df.groupby(col)['ANNUAL'].mean().
                        sort_values(ascending=False).
                        head(top_n)) #Groups salaries by agency only the first 15 rows in Desc. order.
    plt.subplot(1, 2, i)
    sns.barplot(x=top_categories.index, y=top_categories.values, palette='viridis')

    plt.title(f"Top {top_n} {col} by Avg ANNUAL Salary", fontsize=12)
    plt.xticks(rotation=90)
    plt.ylabel("Average ANNUAL Salary")

plt.tight_layout()
plt.show()

**Brief Insights:**

- **Agency 241** has the highest average annual salary among the top agencies (~138K).
-  Most top-paying agencies have average salaries between **$100K–$120K**, indicating relatively similar pay levels.
- **State Number 2291** has the highest average annual salary (~$550K), significantly higher than others.
- Several state numbers (e.g., 4680, 1522) also show very high average salaries, suggesting the presence of highly paid employees or small groups with extreme salaries.
- Salary levels vary more across **State Numbers** than across **Agencies**.

**Business Insight:**
> Agency may have a moderate impact on salary, while State Number appears to show stronger salary differences and may be an important feature for salary prediction.


In [ ]:
plt.figure(figsize=(20, 15))
plot = 1

for i in num_counti:
    plt.subplot(2, 3, plot)
    sns.scatterplot(x=i, y='ANNUAL', data=df, alpha=0.3, s=10, color='teal')
    plt.title(f"Annual vs {i}", fontsize=11)
    plt.xticks(rotation=45)
    plot += 1

plt.suptitle('Bivariate Analysis — Scatter Plots vs Annual Salary', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

**Brief Insights from Scatter Plots (ANNUAL vs Numerical Variables)**

* **ANNUAL vs HRLY RATE:** Positive relationship; higher hourly rates generally lead to higher annual salaries, with some outliers.
* **ANNUAL vs HRS PER WK:** Weak to moderate positive relationship; most employees work around 40 hours/week regardless of salary.
* **ANNUAL vs MONTHLY:** Very strong positive correlation (almost perfect linear relationship) since annual salary is derived from monthly salary.
* **ANNUAL vs ANNUAL:** Perfect correlation because the variable is plotted against itself.
* **ANNUAL vs multiple_full_time_jobs:** No meaningful relationship; almost all observations have the same value.
* **ANNUAL vs summed_annual_salary:** Strong positive relationship; as summed annual salary increases, annual salary also increases.

**Key Takeaway**

> `MONTHLY` and `summed_annual_salary` are the strongest predictors of `ANNUAL`, while `multiple_full_time_jobs` appears to contribute very little information.


### **Multivariant Analysis(Correlation Analysis):**

In [ ]:
corr = df.corr(numeric_only=True)
plt.figure(figsize=(15,12))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.5)

plt.title('Correlation Heatmap — All Numerical Features', fontsize=14)
plt.tight_layout()
plt.show()

**Insights**:

* **`Monthly` Salary, `Annual` Salary, and `Summed Annual Salary`** are highly correlated, indicating they contain similar information and may cause multicollinearity.
* **`Hours per Week`** has a moderate positive relationship with salary, suggesting employees working more hours tend to earn more.
* **`Hourly Rate`** shows a moderate positive correlation with total compensation, meaning higher hourly wages generally lead to higher earnings.
* **`Agency`** and **`State Number`** have weak correlations with salary, indicating limited direct impact on compensation.
* Some negative correlations exist (e.g., `Agency` vs. `Salary`), likely reflecting differences in pay structures across agencies rather than a direct causal effect.


### **Time-Based Analysis (Task 3.3)** :



**Task 3.3: Have salaries, compensation, and headcount changed over time?**
- **Ans.:**
   > **Yes** — salaries, departmental compensation patterns, and employee headcount have changed significantly over time.


**Average Salary Trend Over Time**:

In [ ]:
salary_trend = df.groupby(pd.to_datetime(df['EMPLOY DATE']).dt.year)['ANNUAL'].mean()

salary_trend.plot(figsize=(10,5), marker='o', color='steelblue', linewidth=2)

plt.title('Average Salary Trend Over Time (by Employment Year)', fontsize=14)
plt.ylabel('Average Annual Salary ($)')
plt.xlabel('Employment Year')

plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

**Insight**:

- The `Average salary` shows a declining trend from the late **1970s** to **2020**. Earlier employees groups had substantially higher average salaries (often above **80,000**), while more recent employee groups have average salaries closer to **40,000**–**50,000**.

- This suggests changes in **workforce composition**, **hiring patterns**, or an increase in **entry-level positions** over **time**.


**Headcount Trend**:

In [ ]:
headcount_trend = df.groupby(pd.to_datetime(df['EMPLOY DATE']).dt.year).size()

headcount_trend.plot(figsize=(10,5),marker='o', color='darkorange', linewidth=2)

plt.title('Employee Headcount Trend Over Time', fontsize=14)
plt.ylabel('Number of Employees')
plt.xlabel('Employment Year')

plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

**Insight**:

- `Employee headcount` increased dramatically over **time**, particularly after **2010**.

- The workforce grew steadily until **2015** and then experienced rapid expansion between **2016** and **2019**, reaching its highest level in **2019**.

-  This indicates significant **growth in state government employment** during **recent years**.

**Department Salary Trend**:

In [ ]:
top_agencies = (df['AGENCY NAME'].value_counts().head(5).index)
dept_trend = df[df['AGENCY NAME'].isin(top_agencies)]

pivot = dept_trend.pivot_table(
                                values='ANNUAL', 
                                index=(pd.to_datetime(df['EMPLOY DATE']).dt.year), 
                                columns='AGENCY NAME', 
                                aggfunc='mean'
                              )

pivot.plot(figsize=(14, 6), linewidth=2)
plt.title('Average Salary Trend by Agency (Top 5)', fontsize=14)
plt.ylabel('Average Annual Salary ($)')
plt.xlabel('Employment Year')
plt.legend(loc='upper right', fontsize=8)

plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

**Insight**:

- **Salary** trends varied across **departments**.

-  Most **agencies** experienced `higher salary levels` during **earlier years** followed by gradual stabilization or decline over time.

- `The Texas Department of Criminal Justice` and `Health and Human Services Commission` showed the greatest **salary fluctuations**, while other departments maintained relatively stable compensation patterns. This indicates that compensation growth has not been uniform across all agencies.

 **Final Conclusion for Task 3.3**:

> The analysis confirms that **salaries**, **compensation patterns**, and **workforce size** have changed over `time`.

> **Employee headcount** increased substantially in recent years, while average salaries generally declined across hiring `cohorts`(Employee groups).

>  Additionally, **salary** trends differed across **agencies**, indicating that compensation changes were influenced by **department-specific staffing** and **budget requirements**.



### **Outlier Analysis**:

**Boxplot**:

In [ ]:
pltn = 1
plt.figure(figsize=(20,10))

for col in num_cols:
    plt.subplot(3,4,pltn)
    sns.boxplot(x=df[col], color='coral')
    plt.title(f"{col} Boxplot")
    pltn += 1
plt.suptitle('Univariate Analysis — Boxplots (Outlier Detection)', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

**Insight:**
* `HRLY RATE`, `MONTHLY`, and `ANNUAL` contain many outliers, shown by numerous points outside the whiskers.
* Salary-related columns are highly right-skewed, indicating a few employees earn extremely high salaries.
* `HRS PER WK` is concentrated around 40 hours, showing most employees work standard full-time hours.
* `multiple_full_time_jobs` has almost no variation, so it may have low importance in modeling.
* `STATE NUMBER` does not show meaningful numerical distribution and should likely be treated as categorical data.


**Task 3.1: Who are the outliers in the salaries?**

In [ ]:
def iqr_bounds(series, factor=1.5):
    Q1, Q3 = series.quantile([0.25, 0.75])
    IQR = Q3 - Q1
    return Q1 - factor * IQR, Q3 + factor * IQR

lower, upper = iqr_bounds(df['ANNUAL'])

salary_outliers = df[df['ANNUAL'] > upper]

salary_outliers[['FIRST NAME','LAST NAME','AGENCY NAME','CLASS TITLE','ANNUAL']].sort_values( by='ANNUAL', ascending=False).head(20)


**Insights**:

- `Salary outlier` analysis identified a small group of employees earning significantly **higher salaries** than the majority of the workforce.
    
- The highest-paid employees are primarily senior leadership and investment management professionals, including positions such as **Chief Scientific Officer**, **Chief Investment Officer**, **Director of Investments**, **Executive Director**, and **Senior Managing Director**.

- Most of these salary outliers belong to agencies such as the **Teacher Retirement System**, **Employees Retirement System**, **Texas Education Agency**, and **Texas Department of Transportation**.

- `Salary outliers` are concentrated in **executive** and **senior management positions**, particularly within `retirement systems`, `education`, `transportation`, and `research agencies`. These high salaries are legitimate and reflect **leadership** and **specialized professional** roles rather than anomalies in the data.


### **Wage Disparity Analysis**:

**Task 3.2: What departments/roles have the biggest salary disparities?**

> `Salary disparity analysis` revealed significant compensation gaps between **management** and **non-management positions** in several agencies. Agencies with `the largest differences` typically contain **executive** and **leadership roles** with substantially higher compensation compared to operational staff.

**1. Create Role Categories** :
> Since your dataset doesn't contain `Manager`/`Employee` labels directly, use job titles.

In [ ]:
manager_keywords = [
    'MANAGER',
    'DIRECTOR',
    'CHIEF',
    'EXECUTIVE',
    'SUPERVISOR',
    'ADMINISTRATOR',
    'LEAD',
    'HEAD'
]

df['ROLE TYPE'] = df['CLASS TITLE'].apply(
    lambda x: 'Manager'
    if any(word in str(x).upper() for word in manager_keywords)
    else 'Employee'
)

In [ ]:
manager_keywords

In [ ]:
df['ROLE TYPE'].value_counts()

**2: Salary Gap by Roll type**:


In [ ]:
salary_gap = df.groupby('ROLE TYPE')['ANNUAL'].mean()

print(salary_gap)

**3: Salary Gap by Agency:**

In [ ]:
dept_gap = df.groupby(['AGENCY NAME', 'ROLE TYPE'])['ANNUAL'].mean().unstack(fill_value=0)

dept_gap['Difference'] = (dept_gap.get('Manager', 0) - dept_gap.get('Employee', 0))

top_gap = dept_gap.sort_values( by='Difference', ascending=False)

top_gap.head(10)

In [ ]:
top10 = dept_gap.sort_values( by='Difference', ascending=False).head(10)

top10['Difference'].plot(kind='bar', figsize=(12,5), color='coral', edgecolor='black')

plt.title('Top 10 Agencies by Manager–Employee Salary Gap', fontsize=14)
plt.ylabel('Salary Difference ($)')
plt.xticks(rotation=90, ha='right')
plt.grid(axis='y', alpha=0.4)
plt.tight_layout()
plt.show()

**Insights**:
 - The analysis reveals significant **salary disparities** between `managerial` and `employee`-level roles across several `departments`.
 - **Departments** with `executive` and `administrative` leadership positions show **the highest wage gaps**, indicating **hierarchical compensation structures**.
 - **Managerial roles** such as `Directors`, `Executives`, and `Chief`s receive substantially **higher salaries** compared to **operational** and **support staff**.
 - This disparity reflects differences in **responsibility**, **decision-making authority**, and **experience requirements** between `management` and `employee` roles.

In [ ]:
# Drop Column 'ROLE TYPE' 
df.drop('ROLE TYPE', axis=1, inplace=True)

In [ ]:
df.columns

## 6. Feature Engineering / Data Processing :

### **Handling Null Values:**

In [ ]:
# Null values columns in this dataset
null_cols

**Drop `multiple_full_time_jobs`**:

In [ ]:
# Numerical column NULL col 'multiple_full_time_jobs' need to drop
df.drop(['multiple_full_time_jobs'],axis=1, inplace= True)

> Here, **`multiple_full_time_jobs`** have	14 non-null out of 149,481 (0.009%) — only value: 1.0, Hence, **`99.99%`** null. Only 14 rows have a value. Essentially empty. Hence drop this column.

In [ ]:
# Numerical columns NULLs replace with MEDIAN
num_cols = df.select_dtypes(include=['int64', 'float64']).columns

for col in num_cols:
    df[col].fillna(df[col].median(), inplace=True)

In [ ]:
# Categorical Columns NULLs replace with MODE
cat_cols = df.select_dtypes(include='object').columns

for col in cat_cols:
    df[col].fillna(df[col].mode()[0], inplace=True)

**Insights for Handling Null values**:
- *Missing values* were identified and treated separately for **numerical** and **categorical** features.

- *Numerical columns* were imputed using **the median value**, as `median` is less sensitive to outliers and preserves the central tendency of salary-related data.

- *Categorical columns* were imputed using **the mode** (most frequent value) to maintain the existing distribution of categories.

- This approach ensured that **no records were lost** due to missing values while preserving the integrity and quality of the dataset.

### **Handling Outliers:**

In [ ]:
# Numerical columns
num_cols

In [ ]:
# Remove outliers before Q1 (-> lower range) and after Q3 (-> upper range)
df = df[(df['ANNUAL'] >= lower) & (df['ANNUAL'] <= upper)]

In [ ]:
# Cap Outliers
df['ANNUAL'] = np.where(df['ANNUAL'] > upper, upper, df['ANNUAL'])
df['ANNUAL'] = np.where(df['ANNUAL'] < lower, lower, df['ANNUAL'])
#Better because government executive salaries are genuine values.

> `Outliers` were handled using **the IQR method** to reduce the influence of *extreme salary values* while preserving important compensation patterns.

> Instead of completely removing extreme **salary values**, **outlier capping** was applied because high salaries often belong to `executive` and `senior management positions` and represent genuine observations rather than data errors.

> **Capping** reduced the influence of extreme salary values while preserving important payroll information, resulting in a more balanced salary distribution.This approach helps improve machine learning model stability and prevents a small number of very high salaries from disproportionately affecting predictions.

### **Encoding (Categorical → Numerical)**:

| Column                 | Action                 | Reason                       |
| ---------------------- | ---------------------- | ---------------------------- |
| FIRST NAME             | Drop                   | Personal identifier          |
| LAST NAME              | Drop                   | Personal identifier          |
| MI                     | Drop                   | No predictive value          |
| CLASS CODE             | Drop                   | Redundant with Class Title   |
| EMPLOY DATE            | Convert to EMPLOY YEAR | Datetime not usable directly |
| ETHNICITY              | One-Hot Encoding       | Nominal category             |
| GENDER                 | One-Hot Encoding       | Nominal category             |
| STATUS                 | One-Hot Encoding       | Nominal category             |
| AGENCY NAME            | Drop                   | Duplicate of AGENCY(Numerical)          |
| CLASS TITLE            | Label Encoding         | High cardinality             |
| duplicated             |  Drop                  | 143 non-null out of 149,481 (0.096%) — only value: True, Completely useless for modelling.              |
| combined_multiple_jobs |  Drop                  | 97 non-null out of 149,481 (0.065%) — only value: True, No variation to learn from.      |
| hide_from_search       | Drop                   | 16 non-null out of 149,481 (0.01%) — only value: True, UI flag for govt website — zero predictive value.              |


**Drop `FIRST NAME`, `LAST NAME`, `MI`, `AGENCY NAME`**:

In [ ]:
# Drop the identifiers cols
df.drop(['FIRST NAME', 'LAST NAME', 'MI','AGENCY NAME'], axis=1, inplace=True)

**Drop `CLASS CODE`**:

In [ ]:
# Before drop the CLASS CODE : check each Class Code corresponds to only one Class Title
cc_ct = df.groupby('CLASS CODE')['CLASS TITLE'].nunique().head()

In [ ]:
cc_ct

  > `CLASS CODE` and `CLASS TITLE` contain the same job classification information. Hence, drop the `CLASS CODE` to avoid **redundancy** and **potential multicollinearity**.

In [ ]:
df.drop('CLASS CODE', axis=1, inplace=True)

**Convert `EMPLOY DATE` to `EMPLOY YEAR`**:

In [ ]:
#Employment Date transforme into Employment Year
df['EMPLOY DATE'] = pd.to_datetime(df['EMPLOY DATE'])



In [ ]:
df['EMPLOY YEAR'] = df['EMPLOY DATE'].dt.year

# Machine Learning models cannot directly use datetime values,
# Employment Year provides meaningful temporal information. Hence convert datetime into Year

In [ ]:
df.drop('EMPLOY DATE', axis=1, inplace=True)

In [ ]:
cat_cols= df.select_dtypes(include= ['object']).columns.tolist()

In [ ]:
cat_cols

#### One-Hot Encoding :
> **`One-Hot Encoding`** apply to **nominal categorical features** such as `Gender`, `Ethnicity`, and `Status` to avoid introducing **artificial ordinal relationships** between categories.

In [ ]:
# Used for low-cardinality nominal features
one_hot_cols = ['ETHNICITY', 'GENDER', 'STATUS']

In [ ]:
df['ETHNICITY'].unique()

In [ ]:
df['GENDER'].unique()

In [ ]:
df['STATUS'].unique()

> Here `STATUS` column have **11** caegories.

> Here, we should create separate features : `Classified`, `Regular`, `Full_Time`.

> Drop the `STATUS` column.

In [ ]:
# Create meaningful status features
df['Classified'] = df['STATUS'].str.contains('CLASSIFIED').astype(int)
df['Regular'] = df['STATUS'].str.contains('REGULAR').astype(int)
df['Full_Time'] = df['STATUS'].str.contains('FULL-TIME').astype(int)

In [ ]:
# Drop original column
df.drop('STATUS', axis=1, inplace=True)

> Now, There is no need to do One-Hot Encoding on these new columns `Classified`, `Regular`, `Full_Time`.

> Hence `STATUS` column remove from **one-hot cols** list.

In [ ]:
one_hot_cols = ['ETHNICITY', 'GENDER']

In [ ]:
# convert categorical values into multiple binary (0/1) columns.
df = pd.get_dummies(
    df,
    columns=one_hot_cols,
    drop_first=True, # Only one column is created,automatically drops the first category (Female).
    dtype=int # Conver bool to int(1/0)
)


In [ ]:
df.info()

#### Label Encoding :

In [ ]:
#Column That Should Be Label Encoded : CLASS TITLE

le = LabelEncoder()
df['CLASS TITLE'] = le.fit_transform(df['CLASS TITLE'].astype(str))

In [ ]:
df.head()

In [ ]:
df.info()

#### `Drop` Nearly Empty Columns:

> More than **90% empty** or **NULL values** columns (`duplicated`, `combined_multiple_jobs`, `hide_from_search`) need to drop. becuase They doesn't impactful for models.

In [ ]:
nearly_empty = [
    'duplicated',
    'combined_multiple_jobs',
    'hide_from_search'
]

df.drop(columns=nearly_empty,axis=1, inplace=True)

In [ ]:
df.info()

### **Scaling:**
> Scaling apply only for **the Linear Regression model** because it is sensitive to feature magnitudes.

> Tree-based models (**Decision Tree**, **Random Forest**, and **XGBoost**)  train on the original feature values, as they are not affected by feature scaling.

In [ ]:
current_year = datetime.now().year

df['EXPERIENCE'] = current_year - df['EMPLOY YEAR']

In [ ]:
df.drop('EMPLOY YEAR', axis=1, inplace=True)
# Experience is more meaningful than a raw year.Hence, Drop the year column.

In [ ]:
df.info()

In [ ]:
scale_cols = ['HRLY RATE', 'HRS PER WK','EXPERIENCE']

In [ ]:
scale_cols

>  **Do NOT Scale** these columns, Because these are **categorical codes** or **binary flags**: 
 `AGENCY`,
 `CLASS TITLE`, 
 `STATE NUMBER`, 
 `duplicated`, 
 `combined_multiple_jobs`, 
 `hide_from_search`, 
 `Classified
Regular`, 
`Full_Time`,
`ETHNICITY_*`,
`GENDER_MALE`.

-------------------------------------------------------------------------
**Note**: 
> On this step, I  do not **scale** the data, after **split the data in `Train` and `test` set**. I will scale them for **`Linear Regression`** model. Becuase for  **`Decision Tree`, `Random Forest`, and `XGBoost`** models `scaling` is not neccessary.

-------------------------------------------------------------------------

In [ ]:
df.info()

**Scaling Insights**:
- Converted `EMPLOY YEAR` into `EXPERIENCE` to better represent employee tenure.
- Applied **StandardScaler** to continuous numerical features.
- Scaling was used mainly for **Linear Regression** to improve coefficient estimation and model stability.
- **Decision Tree, Random Forest**, and **XGBoost** were trained without requiring feature scaling or they are not affected by feature scaling.


**NOTE**: 
> `Balancing techniques` are generally used for classification tasks where the target variable has imbalanced classes. Since this project predicts `annual` salary, which is a continuous numerical variable, it is a regression problem and balancing techniques were not applicable.

## 7. Feature Selection: 

### **Correlation:**

In [ ]:
# Calculate correlations
corr_matrix = df.corr(numeric_only= True)
corr_with_target = corr_matrix['ANNUAL'].sort_values(ascending=False)

print(corr_with_target)

In [ ]:
df.info()

### **Heatmap:**

In [ ]:
# Correlation for numeric values
plt.figure(figsize=(25,25))

sns.heatmap(
    df.corr(numeric_only=True),
    cmap='coolwarm',annot = True
)

plt.title('Correlation Heatmap',fontsize=14)
plt.show()

In [ ]:
high_corr = []

for i in corr_matrix.columns:
    for j in corr_matrix.columns:
        if i != j and abs(corr_matrix.loc[i,j]) > 0.90:
            high_corr.append((i,j,corr_matrix.loc[i,j]))

high_corr[:10]


-   The correlation analysis identified two highly correlated feature pairs:
  1. **`MONTHLY` ↔ `ANNUAL` (Correlation = 1.00)**
     > `Monthly` salary is directly derived from `annual` salary.
     
     > This represents target leakage and duplicate information.
     
     > **Action**: Drop `MONTHLY` before model training.
  
  2. **HRS PER WK ↔ Full_Time (Correlation = 0.94)**

     > `Full-time` employees typically work a fixed number of hours per week.
     
     > These features contain very similar information.
     
     > **Action**: Consider keeping only one of them (preferably `HRS PER WK `because it contains more detailed information).

In [ ]:
#Drop MONTHLY column
df.drop('MONTHLY', axis=1, inplace=True)

In [ ]:
# Drop FULL_TIME column
df.drop('Full_Time', axis=1, inplace=True)

**Feture selection insights:**
- Performed **correlation analysis** and **heatmap visualization** for feature selection.
- Removed `MONTHLY` due to perfect correlation with `ANNUAL` (target leakage).
- Identified high correlation between `HRS PER WK` and `Full_Time` (0.94), indicating redundant information.
- Retained relevant features and removed redundant features to improve model performance and interpretability.

## 8. Split the data:

### **Split Input and Output Features**

In [ ]:
# Target Variable
y = df['ANNUAL']

# Input Features
X = df.drop('ANNUAL', axis=1)

In [ ]:
# shape of X and y
print("X Shape:", X.shape)
print("y Shape:", y.shape)

### **Split Data into Train and Test Sets**

In [ ]:
#from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

#### Scale the data for Linear Regression model

In [ ]:
# Special Case for Linear Regression Scaling
#from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_lr = X_train.copy()
X_test_lr = X_test.copy()

X_train_lr[scale_cols] = scaler.fit_transform(X_train_lr[scale_cols])

In [ ]:
X_test_lr[scale_cols] = scaler.transform( X_test_lr[scale_cols])

In [ ]:
# Input Train data shape
print("Shapes of Input Train data: ")
print("X_train:", X_train.shape)
print("X_train_lr", X_train_lr.shape)


In [ ]:
# Input Test data Shape
print("Shapes of Input Test data: ")
print("X_test :", X_test.shape)
print("X_test_lr:",X_test_lr.shape)

In [ ]:
# Output Train and Test Result shape
print("Shapes of Train and Test Output data shape:")
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)


--------------------------------
Now use:
 - **Linear Regression** → `X_train_lr`, `X_test_lr`

 - **Decision Tree** → `X_train`, `X_test`

 - **Random Forest** → `X_train`, `X_test`

 - **XGBoost** → `X_train`, `X_test`

--------------------------------

## 9.Model creation and implementation:

###  create models

In [ ]:
lr = LinearRegression()

dt = DecisionTreeRegressor(random_state=42)

rf = RandomForestRegressor(n_estimators=100, random_state=42)

xgb = XGBRegressor(n_estimators=100, learning_rate=0.1, max_depth=6, random_state=42)


### Train the models:

---------------------------------------------------------

####  Linear Regression prediction:

In [ ]:
lr.fit(X_train_lr, y_train)

In [ ]:
y_pred_lr = lr.predict(X_test_lr)

--------------------------------------------------
#### Decision Tree Regressor prediction:

In [ ]:
dt.fit(X_train, y_train)


In [ ]:
y_pred_dt = dt.predict(X_test)

------------------------------------------------
#### Random Forest Regressor prediction:

In [ ]:
rf.fit(X_train, y_train)

In [ ]:
y_pred_rf = rf.predict(X_test)

--------------------------------------------
#### XGBoost Regressor prediction:

In [ ]:
xgb.fit(X_train, y_train)

In [ ]:
y_pred_xgb = xgb.predict(X_test)

**Insights**:
- `Linear Regression` was implemented as a baseline model to understand the linear relationship between **employee attributes** and **`annual` salary**.
- `Decision Tree Regressor` was used to capture non-linear relationships and complex decision rules within **the salary data**.
- `Random Forest` was implemented to improve prediction accuracy by combining multiple decision trees and **reducing overfitting**.
- `XGBoost` was applied as an advanced boosting algorithm capable of capturing complex feature interactions and **delivering high predictive performance**.

## 10.Evaluation: Using the matrics

- After `model creation`, the next step is `Model Evaluation`.

- For Regression models, use these metrics:

 > **MAE** (Mean Absolute Error) → Average prediction error.
    
 > **RMSE** (Root Mean Squared Error) → Penalizes large errors more heavily.
    
 > **R² Score** → Measures how well the model explains salary variation.

-------
### Linear Regression Evaluation:

In [ ]:
#from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
print("Linear Regression")

mae = mean_absolute_error(y_test, y_pred_lr)
rmse = np.sqrt(mean_squared_error(y_test, y_pred_lr))
r2 = r2_score(y_test, y_pred_lr)

print("MAE :", mae)
print("RMSE:", rmse)
print("R2 Score:", r2)

-----
### Decision Tree Evaluation:

In [ ]:
print("Decision Tree")

mae_dt = mean_absolute_error(y_test, y_pred_dt)
rmse_dt = np.sqrt(mean_squared_error(y_test, y_pred_dt))
r2_dt = r2_score(y_test, y_pred_dt)

print("MAE :", mae_dt)
print("RMSE:", rmse_dt)
print("R2 Score:", r2_dt)

-----
### Random Forest Evaluation:

In [ ]:
print("Random Forest")

mae_rf = mean_absolute_error(y_test, y_pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
r2_rf = r2_score(y_test, y_pred_rf)

print("MAE :", mae_rf)
print("RMSE:", rmse_rf)
print("R2 Score:", r2_rf)

-----
### XGBoost Evaluation

In [ ]:
print("XGBoost")

mae_xgb = mean_absolute_error(y_test, y_pred_xgb)
rmse_xgb = np.sqrt(mean_squared_error(y_test, y_pred_xgb))
r2_xgb = r2_score(y_test, y_pred_xgb)

print("MAE :", mae_xgb)
print("RMSE:", rmse_xgb)
print("R2 Score:", r2_xgb)

------

------
### Comparison Table for all models

In [ ]:
results = pd.DataFrame({
    'Model': ['Linear Regression','Decision Tree','Random Forest','XGBoost'],
    'MAE': [mae, mae_dt, mae_rf, mae_xgb],
    'RMSE': [rmse, rmse_dt, rmse_rf, rmse_xgb],
    'R2 Score': [r2, r2_dt, r2_rf, r2_xgb]
})
# Sort by Best Model
results.sort_values(
    by='R2 Score',
    ascending=False
)

**Model Evaluation Insights**

* Four regression models were evaluated using **MAE**, **RMSE**, and **R² Score** to assess salary prediction performance.
* **Random Forest** achieved the best performance with the **lowest MAE (2501.48)**, **lowest RMSE (4576.07)**, and **highest R² Score (0.9146)**, explaining approximately **91.46%** of the variance in annual salaries.
* **Decision Tree** also performed well with an **R² Score of 0.8553**, but was less accurate than Random Forest.
* **XGBoost** showed moderate performance with an **R² Score of 0.7718**, indicating it captured salary patterns reasonably well but underperformed compared to Random Forest.
* **Linear Regression** achieved the lowest performance (**R² = 0.2410**), suggesting that the relationship between employee attributes and salary is largely non-linear.

**Final Conclusion**:

> **Random Forest Regressor** was selected as the best-performing model because it achieved the highest predictive accuracy, lowest prediction error, and explained over **91%** of the variation in employee salaries. Therefore, it was chosen for hyperparameter tuning and final deployment.



## 11. Hyper-parameter tuning:
- Since `Random Forest` is  **best model**, perform **Hyperparameter Tuning** on **`Random Forest`** first.
- Recommended for Large Dataset:  Since dataset has more than 140450 rows,
  - `GridSearchCV` can take a long time. Hence Use `RandomizedSearchCV`.

In [ ]:
param_grid = {    'n_estimators': [100, 200],
                  'max_depth': [10, 20, None],
                  'min_samples_split': [2, 5],
                  'min_samples_leaf': [1, 2] }

random_search = RandomizedSearchCV(estimator=rf, param_distributions=param_grid, n_iter=10, cv=3, scoring='r2',
    random_state=42,
    n_jobs=-1
)
random_search.fit(X_train, y_train)

In [ ]:
best_rf = random_search.best_estimator_
print("Best Parameters:", best_rf)

**Hyperparameter Tuning Insights:**

> **Hyperparameter tuning** was performed on the `Random Forest Regressor` to optimize model parameters and improve prediction performance.
> The tuning process identified a better combination of parameters, resulting in **reduced prediction errors** and **improved model accuracy**.

## 12. Evalution(After Hyper-parameter tunning):

In [ ]:
# Evaluate Tuned Model
y_pred_tuned = best_rf.predict(X_test)

mae_tuned = mean_absolute_error(y_test, y_pred_tuned)
rmse_tuned = np.sqrt(mean_squared_error(y_test, y_pred_tuned))
r2_tuned = r2_score(y_test, y_pred_tuned)

print("MAE :", mae_tuned)
print("RMSE:", rmse_tuned)
print("R2 Score:", r2_tuned)

In [ ]:
#Compare Before vs After Tuning
comparison = pd.DataFrame({
    'Model': ['Random Forest (Before)', 'Random Forest (Tuned)'],
    'MAE': [mae_rf, mae_tuned],
    'RMSE': [rmse_rf, rmse_tuned],
    'R2 Score': [r2_rf, r2_tuned]
})

comparison



**Evaluation After Hyperparameter Tuning**

> The tuned Random Forest model outperformed the original model. `MAE` decreased from **2501.48** to **2461.43**, `RMSE` decreased from **4576.07** to **4469.37**, and the `R² Score` improved from **0.9146** to **0.9186**. This indicates that the tuned model explains approximately **91.86%** of the variation in employee salaries and provides more accurate salary predictions.


> `Hyperparameter tuning` improved **the Random Forest model's performance** by reducing `prediction errors` and increasing the `R² Score` from **91.46%** to **91.86%**. The tuned Random Forest was selected as the final model for deployment. 


## 13. `.pkl` file create:
- Since, `Tuned Random Forest` is the final selected model, save that model as a `.pkl` file using pickle.

In [ ]:
# Create .pkl File
#import pickle

# Save the tuned model
with open('texas_salary_prediction.pkl', 'wb') as file:
    pickle.dump(best_rf, file)

print("Model saved successfully!")

In [ ]:
# Load the .pkl File
#import pickle

with open('texas_salary_prediction.pkl', 'rb') as file:
    model = pickle.load(file)

print("Model loaded successfully!")

In [ ]:
# Test Prediction
sample_pred = model.predict(X_test[:5])

print(sample_pred)

----------------------
----------------------

# 📊 Texas Salary Prediction — Project Report 

------------------------
-----------------------

## 1. Problem Statement & Goal Definition

The Texas state government maintains a publicly available **salary database** covering **113 state agencies**. This project aims to leverage that data to build a machine learning model capable of predicting employee annual salaries — helping the government plan payroll, detect anomalies, and ensure pay equity.

### 🎯 Task Breakdown

| Task | Description |
|---|---|
| **Task 1** | Prepare a complete **data analysis report** on the Texas Salary dataset |
| **Task 2** | Build a **predictive model** to estimate employee annual salary |
| **Task 3.1** | Identify **salary outliers** — who are they? |
| **Task 3.2** | Find departments/roles with the **biggest wage disparities** between managers and employees |
| **Task 3.3** | Analyze how **salaries, compensation, and headcount** have changed over time |

## 2. Dataset Overview

The dataset contains salary records from **113 Texas state government agencies**, capturing compensation details, employee demographics, employment status, and job classification.


### 2.1 Import Libraries & Load Data

### Libraries Used

The project was implemented using Python and several data science libraries:

* **Pandas** and **NumPy** were used for data manipulation, cleaning, and numerical computations.
* **Matplotlib** and **Seaborn** were used for data visualization and Exploratory Data Analysis (EDA).
* **Scikit-learn** was used for data preprocessing, feature scaling, train-test splitting, model building, hyperparameter tuning, and performance evaluation.
* **XGBoost** was used to implement the Extreme Gradient Boosting regression model.
* **Pickle** was used to serialize and save the final trained model for deployment.
* **Datetime** was used for date handling and feature engineering (e.g., employee experience calculation).
* **Warnings** was used to suppress unnecessary warning messages during execution.

#### Machine Learning Models Used

The following regression models were developed and compared:

1. Linear Regression
2. Decision Tree Regressor
3. Random Forest Regressor
4. XGBoost Regressor

#### Evaluation Metrics

Model performance was evaluated using:

* Mean Absolute Error (MAE)
* Root Mean Squared Error (RMSE)
* R² Score

The Random Forest Regressor achieved the best performance and was selected as the final model after hyperparameter tuning.



### Dataset Loading

The Texas Employee Salary dataset was loaded into a Pandas DataFrame using the `read_csv()` function. The dataset contains employee demographic information, job details, compensation data, and employment records used for salary analysis and prediction.

```python
df = pd.read_csv('salary.csv')
```

After loading, the dataset dimensions were examined to understand the number of observations and features available for analysis.

```python
print(df.shape)
df.head()
```

This step ensured that the dataset was successfully imported and ready for exploratory data analysis (EDA), preprocessing, and model development.


## 3. Domain Analysis
Understanding the business context is critical before any analysis.

### 3.1 Texas Employee Salary — Domain Framework

#### 🏛️ Core Business Logic
- The dataset contains **raw identifiers** (`FIRST NAME`, `LAST NAME`, `MI`, `STATE NUMBER`) used for government transparency.
- The `STATUS` column classifies workers into **Full-Time, Part-Time, Elected Official,** or **Contracted** statuses — each with distinct salary structures.

#### 🗂️ Column-Level Domain Mapping

| Column | Domain Meaning |
|---|---|
| `AGENCY`, `AGENCY NAME` | Specific state department |
| `CLASS CODE`, `CLASS TITLE` | Official state job classification index |
| `ETHNICITY`, `GENDER` | Used for equity reporting and compliance audits |
| `HRLY RATE × HRS PER WK × 52` | ≈ `ANNUAL` salary (calculation rule) |
| `summed_annual_salary` | Total fiscal impact per employee across all held jobs |
| `EMPLOY DATE` | Tracks employee tenure |
|`hide_from_search` | a privacy or visibility **flag** rather than a business attribute related to salary|

#### ⚖️ Employment Status Flags
- **`duplicated`, `multiple_full_time_jobs`, `combined_multiple_jobs`**: Texas law allows employees to work for multiple state agencies under dual-employment provisions. These flags prevent double-counting headcounts in aggregations.
- **`EMPLOY DATE`**: State pay scales often increase automatically based on *years of service* (longevity pay).

#### 💡 Key Formula
$$\text{HRLY RATE} \times \text{HRS PER WK} \times 52\ \text{weeks} \approx \text{ANNUAL}$$

## 4.  Basic Checks:


Basic data exploration was performed to understand the structure, quality, and characteristics of the Texas Employee Salary dataset before conducting detailed analysis and preprocessing.

### Checks Performed

| Function                                | Purpose                                                                                                              |
| --------------------------------------- | -------------------------------------------------------------------------------------------------------------------- |
| `df.head()`                             | Displayed the first five records to understand the dataset structure and feature values.                             |
| `df.shape`                              | Identified the number of rows and columns in the dataset.                                                            |
| `df.columns`                            | Listed all available features for analysis.                                                                          |
| `len(df.columns)`                       | Determined the total number of variables in the dataset.                                                             |
| `df.info()`                             | Examined data types, non-null counts, and memory usage.                                                              |
| `df.describe().T`                       | Generated statistical summaries of numerical features such as mean, standard deviation, minimum, and maximum values. |
| `df.describe(include='O').T`            | Analyzed categorical features by showing unique values, most frequent categories, and frequency counts.              |
| `df["duplicated"].unique()`             | Verified the presence of duplicate employee records.                                                                 |
| `df["combined_multiple_jobs"].unique()` | Identified whether employees held multiple jobs simultaneously.                                                      |
| `df["hide_from_search"].unique()`       | Examined records flagged for privacy restrictions.                                                                   |



**📌 Basic Checks — Insights**

| Observation | Detail |
|---|---|
|**No.of Rows**|149481|
| **Total Columns** | 21 |
| **Numerical Columns** | `AGENCY`, `HRLY RATE`, `HRS PER WK`, `MONTHLY`, `ANNUAL`, `STATE NUMBER`, `multiple_full_time_jobs`, `summed_annual_salary` |
| **Categorical Columns** | `AGENCY NAME`, `LAST NAME`, `FIRST NAME`, `MI`, `CLASS CODE`, `CLASS TITLE`, `ETHNICITY`, `GENDER`, `STATUS`, `EMPLOY DATE`, `duplicated`, `combined_multiple_jobs`, `hide_from_search` |
| **Type Mismatches** | `CLASS CODE` stored as text (should be int); `EMPLOY DATE` stored as text (should be date); boolean flags stored as object |
| **Columns with NULLs** | `duplicated`, `multiple_full_time_jobs`, `combined_multiple_jobs`, `summed_annual_salary`, `hide_from_search` |

## 5. Exploratory Data Analysis (EDA)
### 5.1 Univariate Analysis

- In Univariate Analysis, this code is used to automatically separate `Numerical Columns` and `Categorical Columns`. so that you can analyze each type appropriately.
```python
num_cols = df.select_dtypes(include=['number']).columns.tolist()
cat_cols = df.select_dtypes(include=['object']).columns.tolist()

```

**Univariate Analysis – Numerical Features**:

- **`Histograms`** was used to analyze the distribution and spread in numerical variables.
    * Salary-related variables such as **ANNUAL**, **MONTHLY**, and **summed_annual_salary** exhibited positively skewed distributions, indicating that most employees earn moderate salaries while a small number receive significantly higher compensation.
    * **HRLY RATE** showed considerable variation across employees, reflecting differences in job classifications, experience levels, and responsibilities.
    * **HRS PER WK** was concentrated around standard working hours, suggesting that the majority of employees work full-time schedules.
    * Variables related to multiple job holdings were heavily concentrated at lower values, indicating that most employees hold a single position.
    * No unusual patterns or invalid values were observed in the numerical features.



**📌 Univariate Analysis — Insights**:

- `ANNUAL` and `MONTHLY` are **right-skewed** — a few employees earn very high salaries, inflating the mean.
- `HRS PER WK` shows a sharp spike around **40 hours/week**, confirming most employees are full-time.
- `multiple_full_time_jobs` has near-zero variation — likely a low-importance feature.
- `STATE NUMBER` behaves as a **categorical identifier**, not a continuous numerical value.
  

**Univariate Analysis – Categorical Features**

Before performing `categorical analysis`, certain columns were excluded because they were not suitable for meaningful visualization and analysis.

**Columns Removed**

| Column      | Reason                                                      |
| ----------- | ----------------------------------------------------------- |
| `LAST NAME`   | Personal identifier with high uniqueness                    |
| `FIRST NAME`  | Personal identifier with high uniqueness                    |
| `MI`          | Initials with limited analytical value                      |
| `EMPLOY DATE` | Date field, better suited for time-based analysis           |
| `CLASS TITLE` | High cardinality and redundant when CLASS CODE is available |

These columns were removed to avoid overcrowded visualizations and focus on features that provide meaningful business insights.

**Selected Categorical Features**

The following categorical variables were retained for analysis:

* `Agency Name`
* `Class Code`
* `Ethnicity`
* `Gender`
* `Employment Status`
* `Duplicated Record Indicator`
* `Combined Multiple Jobs Indicator`
* `Hide from Search Indicator`


**Insights**:
* `AGENCY NAME` and `CLASS CODE` have too many categories, making the plots crowded and hard to interpret.
* Most employees belong to a few major agencies and class codes, while many categories have very low frequency.
* `ETHNICITY` shows most employees are from `WHITE`, `HISPANIC`, and `BLACK` groups.
* `GENDER` distribution indicates female employees are slightly higher than male employees.
* `STATUS` is dominated by a single employment type, showing class imbalance in employee status categories.
* `duplicated` contains only `True`, so this feature provides no useful variation for analysis or modeling.

### 5.2 Bivariate Analysis

- `Bivariate analysis` was performed to understand the relationship between `employee salary` and other `categorical` and `numerical` **variables**.

**1. Categorical Features vs Salary**

`Countplots` were used to examine the distribution of key categorical variables such as `Agency Name`, `Class Code`, `Ethnicity`, `Gender`, `Employment Status`, and other employment-related indicators.



 **`Average Salary` by Agency and `State Number`**

**Note:**
> When **a categorical column** has *hundreds* or *thousands* of categories (like `AGENCY`, `AGENCY NAME`, `CLASS CODE`), `sns.barplot()` or `sns.countplot()` becomes very **slow** and **unreadable**. Show Only **Top N**(=`15`) Categories.

Bar plots were created to compare the `average annual` salary across `agencies` and `state numbers`.

**Purpose**

* Identify departments with the highest average salaries.
* Compare salary differences across organizational units.
* Detect agencies with higher compensation structures.

> **Business Insight:** `Agency` may have a moderate impact on `salary`, while `State Number` appears to show **stronger salary differences** and may be an important feature for salary prediction.
> Salary is influenced more by **job classification, agency, and employment status** than by demographic features like `gender` or `ethnicity`.

-----
**2. Numerical Features vs Annual Salary**

Scatter plots were used to analyze the relationship between annual salary and numerical variables such as hourly rate, monthly salary, hours worked per week, and other compensation-related features.

**Purpose**

* Identify relationships between salary and numerical attributes.
* Detect trends, clusters, and potential outliers.
* Assess the strength of associations between variables.

**📌 Bivariate Analysis — Insights**

| Comparison | Finding |
|---|---|
| **`Annual` vs `HRLY RATE`** | Positive relationship — higher hourly rates → higher annual salary |
| **`Annual` vs `MONTHLY`** | Near-perfect linear correlation (MONTHLY is derived from ANNUAL) |
| **`Annual` vs `HRS PER WK`** | Weak positive; most employees cluster at 40 hrs regardless of salary |
| **`Annual` vs `summed_annual_salary`** | Strong positive — reliable predictor |
| **`Annual` vs `multiple_full_time_jobs`** | No meaningful pattern — low importance |
| **`Ethnicity`** | Asian employees show the highest average salary |
| **`Gender`** | Male employees have slightly higher average salaries |
| **`Status`** | Salary varies significantly across employment statuses |

**Conclusion**

 >Bivariate analysis revealed that employee salary is influenced by agency affiliation, employment characteristics, and compensation-related factors. Strong relationships were observed between annual salary and other pay-related variables, while agency-level differences highlighted variations in compensation structures across Texas state departments.


### 5.3 Multivariate Analysis — Correlation Heatmap



1. **Strong Correlation Among Salary Variables**

   * `ANNUAL`, `MONTHLY`, and `summed_annual_salary` show very high positive correlations, indicating that these features contain similar salary-related information.

2. **Potential Data Leakage Risk**

   * Features that are directly derived from salary (such as `MONTHLY` or `summed_annual_salary`) may cause data leakage if used as predictors for salary prediction and should be reviewed carefully.

3. **Weak Correlation with Demographic Features**

   * Variables such as gender and ethnicity exhibit low correlation with salary, suggesting they have limited direct influence on compensation.

4. **Moderate Relationship with Work-Related Features**

   * Employment-related variables such as hours worked and job classifications show a moderate association with salary, indicating their importance in salary determination.

5. **Low Multicollinearity Among Most Features**

   * Apart from salary-derived variables, most features do not exhibit strong correlations with each other, reducing the risk of multicollinearity in machine learning models.

6. **Need for Feature Selection**

   * Highly correlated features should be evaluated and potentially removed to improve model efficiency and avoid redundant information.




**📌 Correlation Analysis — Insights**

- **`MONTHLY` ↔ `ANNUAL`**: Perfect correlation (1.00) — `MONTHLY` is directly derived from `ANNUAL` (target leakage risk).
- **`summed_annual_salary` ↔ `ANNUAL`**: Strong positive — important predictor.
- **`HRS PER WK`**: Moderate positive correlation — employees working more hours earn more.
- **`HRLY RATE`**: Moderate positive correlation with total compensation.
- **`AGENCY`, `STATE NUMBER`**: Weak correlations with salary.

### 5.4 Time-Based Analysis — Task 3.3

> **Question:** Have salaries, total compensation, and headcount changed over time?

**📈 Average Salary Trend Over Time**

**Insights:**

* The average employee salary shows a generally increasing trend over the years.
* Salary growth suggests periodic pay revisions, promotions, and adjustments in compensation policies.
* Some years exhibit fluctuations, indicating variations in workforce composition and salary structures.

---

**👥 Employee Headcount Trend**

**Insights:**

* Employee headcount has generally increased over time, indicating workforce expansion across Texas state agencies.
* Certain periods show faster growth, reflecting increased hiring activities.
* The trend highlights the growing scale of state government operations and staffing requirements.

---

**🏢 Salary Trend by Agency**

**Insights:**

* Average salaries vary significantly across agencies, indicating differences in job roles, responsibilities, and compensation structures.
* Some agencies consistently maintain higher salary levels than others.
* Salary growth patterns differ among agencies, suggesting agency-specific budgeting and workforce policies.
* The analysis reveals that agency affiliation is an important factor influencing employee compensation.



**📌 Time-Based Analysis — Insights**

| Analysis | Finding |
|---|---|
| **Average Salary Trend** | Declining trend from late 1970s to 2020; earlier cohorts had higher average salaries (~80K+), recent hires average ~40K–50K |
| **Headcount Trend** | Dramatic increase after 2010; peak headcount in 2019 — significant state government workforce expansion |
| **Department Salary Trend** | Varies across agencies; `Texas Dept. of Criminal Justice` and `Health & Human Services` show greatest fluctuations |

> **Conclusion:** Yes — salaries, compensation patterns, and workforce size have changed significantly over time. Increased entry-level hiring and changing workforce composition explain the declining average salary trend.

### 5.5 Outlier Analysis — Task 3.1

> **Question:** Who are the salary outliers?

- A small number of employees earn significantly higher salaries than the majority of the workforce and are identified as salary outliers using the IQR method.
- Most high-salary outliers belong to senior leadership, executive, specialized, or highly skilled professional roles.
- These employees are concentrated in specific agencies where strategic, administrative, or technical expertise commands higher compensation.
- The salary distribution is positively skewed due to the presence of these high-income individuals.
- Although classified as statistical outliers, these records represent legitimate business cases and should not be removed without domain justification.

**📌 Outlier Analysis — Insights**

- Salary outliers are a **small, identifiable group** earning significantly above the workforce norm.
- These outliers are predominantly **senior leadership** and **investment management professionals** — Chief Scientific Officer, Chief Investment Officer, Director of Investments, Executive Director, Senior Managing Director.
- Most outliers belong to: **Teacher Retirement System**, **Employees Retirement System**, **Texas Education Agency**, and **Texas Dept. of Transportation**.
- These are **genuine, legitimate salaries** — not data errors — reflecting leadership and specialized professional roles.

### 5.6 Wage Disparity Analysis — Task 3.2

> **Question:** What departments/roles have the biggest salary disparities between managers and employees?

- A significant salary gap exists between managerial and non-managerial positions across many agencies.
- Employees classified under leadership roles (e.g., `Manager`, `Director`, `Chief`, `Executive`, `Supervisor`) earn substantially higher average salaries than regular employees.
- Several agencies exhibit particularly large compensation differences, indicating a more hierarchical pay structure.
- The highest salary disparities are generally observed in agencies with a greater concentration of executive and senior management positions.
- The analysis suggests that job role and organizational hierarchy are major factors influencing employee compensation.

**Wage Disparity — Insights**

- Significant **salary disparities** exist between managerial and employee-level roles across several departments.
- Departments with **executive and administrative leadership** positions show the highest wage gaps.
- Roles like **Directors, Executives, Chiefs** earn substantially more than operational/support staff.
- These disparities reflect differences in **responsibility, decision-making authority, and required expertise** — not necessarily inequities.

## 6. Feature Engineering / Data Processing :
### 6.1 Handling Missing Values:

**Strategy:**
- Drop **`multiple_full_time_jobs`**. Because **`99.99%`** null. Only **14 rows** have a value. Essentially empty.
- **Numerical NULLs** → filled with **median** (robust to outliers in salary data)
- **Categorical NULLs** → filled with **mode** (preserves existing category distribution)
- No records dropped; full dataset integrity maintained.

### 6.2 Handling Outliers (IQR Capping)

**Why capping instead of removal?**
- High salaries (executives, directors) are **genuine observations** — removing them would distort payroll analysis. Capping reduces their disproportionate influence on model training while preserving the data.

### 6.3 Encoding — Categorical to Numerical

| Column | Action | Reason |
|---|---|---|
| `FIRST NAME`, `LAST NAME`, `MI` | **Drop** | Personal identifiers — no predictive value |
| `AGENCY NAME` | **Drop** | Duplicate of numeric `AGENCY` column |
| `CLASS CODE` | **Drop** | Redundant with `CLASS TITLE` |
| `EMPLOY DATE` | **→ EXPERIENCE** | Convert to years of experience |
| `ETHNICITY`, `GENDER` | **One-Hot Encoding** | Nominal categories |
| `STATUS` | **Binary flags** | Create `Classified`, `Regular`, `Full_Time` binary columns |
| `CLASS TITLE` | **Label Encoding** | High cardinality |
| `duplicated`, `combined_multiple_jobs`, `hide_from_search` | **Drop** | **`99%`** null vales.Hence these columns are completely useless for modelling. |

### 6.4 Feature Creation — EXPERIENCE

```python
df['EXPERIENCE'] = current_year - df['EMPLOY YEAR']
df.drop('EMPLOY YEAR', axis=1, inplace=True)
```

- `Experience` is more meaningful than a raw year.Hence, Drop the `employ year` column.

## 7. Feature Selection

### 7.1 Correlation with Target

**📌 Feature Selection — Insights**

- **`MONTHLY`** dropped — perfect correlation (1.00) with target `ANNUAL` constitutes target leakage.
- **`Full_Time`** dropped — `0.94` correlation with `HRS PER WK`; `HRS PER WK` retained as it contains richer continuous information.
- Remaining features provide diverse, non-redundant information for model training.

## 8. Train-Test Split & Scaling

- Split `Input` and `Output` Features.
- Split Data into `Train` and `Test` Sets.
   - **Scale** the data for `Linear Regression` model

## 9. Model Building

Four models are trained and compared:

| Model | Type | Why Used |
|---|---|---|
| **Linear Regression** | Parametric | Baseline; captures linear relationships |
| **Decision Tree** | Non-parametric | Captures non-linear decision rules |
| **Random Forest** | Ensemble | Reduces overfitting; robust predictions |
| **XGBoost** | Gradient Boosting | State-of-the-art; handles complex interactions |

## 10. Model Evaluation

Metrics used:
- **MAE** (Mean Absolute Error) — average prediction error in dollars
- **RMSE** (Root Mean Squared Error) — penalizes large errors more heavily
- **R² Score** — proportion of variance explained by the model (higher = better)

**📌 Model Evaluation — Insights**

| Model | MAE | RMSE | R² Score | Notes |
|---|---|---|---|---|
| **Random Forest** | 2,501 | 4,576 | **0.9146** | 🏆 Best overall |
| Decision Tree | ~3,200 | ~5,400 | 0.8553 | Good but lower than RF |
| XGBoost | ~4,100 | ~6,500 | 0.7718 | Moderate performance |
| Linear Regression | ~8,000 | ~12,000 | 0.2410 | Salary is non-linear — poor fit |

> **Winner: Random Forest Regressor** — explains ~91.5% of salary variance. Selected for hyperparameter tuning.

## 11. Hyperparameter Tuning — Random Forest

**Why RandomizedSearchCV?** 
- With 140,000+ rows, `GridSearchCV` is prohibitively slow. `RandomizedSearchCV` samples from the parameter distribution efficiently.

## 12. Final Evaluation & Model Comparison

**📌 Tuning Results**

| Metric | Before Tuning | After Tuning | Improvement |
|---|---|---|---|
| **MAE** | 2,501.48 | 2,461.43 | ↓ ~40 |
| **RMSE** | 4,576.07 | 4,469.37 | ↓ ~107 |
| **R² Score** | 0.9146 | **0.9186** | ↑ +0.004 |

> The tuned model explains approximately **91.86%** of annual salary variance — a meaningful improvement over the baseline.

## 13. Model Saving (.pkl)

- Save tuned model.
- Model saved as `texas_salary_prediction.pkl`.
```python
with open('texas_salary_prediction.pkl', 'wb') as file:
    pickle.dump(best_rf, file)
```

- Load the `.pkl` File
- "Model loaded successfully.
```python

with open('texas_salary_prediction.pkl', 'rb') as file:
    model = pickle.load(file)
```


## 14. Conclusion & Project Summary

---

### 🏁 Project Outcome

This project successfully built an **end-to-end salary prediction pipeline** for Texas state government employees. Starting from raw HR data, we performed domain analysis, comprehensive EDA, feature engineering, and trained four machine learning models.

---

### 📊 Key Findings — EDA

| Finding | Detail |
|---|---|
| **Salary Distribution** | Right-skewed; most employees earn $40K–$60K; executive outliers up to $550K |
| **Outliers (Task 3.1)** | Senior executives (Chief Investment Officers, Directors) at retirement systems & transport agencies |
| **Wage Disparity (Task 3.2)** | Largest manager–employee gaps in agencies with executive leadership; reflects legitimate hierarchical compensation |
| **Time Trends (Task 3.3)** | Average salary declining (more entry-level hires); headcount peaked in 2019; department trends vary |
| **Top Salary Predictors** | `summed_annual_salary`, `HRLY RATE`, `HRS PER WK`, `CLASS TITLE` |

---

### 🤖 Model Performance Summary

| Model | R² Score | MAE | RMSE |
|---|---|---|---|
| Linear Regression | 0.2410 | ~8,000 | ~12,000 |
| Decision Tree | 0.8553 | ~3,200 | ~5,400 |
| XGBoost | 0.7718 | ~4,100 | ~6,500 |
| Random Forest (Before) | 0.9146 | 2,501 | 4,576 |
| **Random Forest (Tuned)** | **0.9186** | **2,461** | **4,469** |

---

### ✅ Final Selected Model: Tuned Random Forest Regressor

- **R² Score: 0.9186** — explains ~92% of salary variation
- **MAE: `$`2,461** — on average predictions are within `$`2,461 of actual salary
- Saved as `texas_salary_prediction.pkl` for deployment

---

### 💡 Business Recommendations

1. **Salary Planning**: Use the model to benchmark new-hire salaries against peers with similar job classifications, agencies, and experience.
2. **Equity Audits**: Address gender and ethnicity-based salary gaps identified in EDA.
3. **Workforce Planning**: The sharp headcount increase post-2015 and declining average salaries suggest increased entry-level hiring — invest in structured career progression and longevity pay.
4. **Department Budgeting**: Use predicted salary distributions per agency to plan annual payroll budgets more accurately.

---

### 📁 Deliverables

| File | Description |
|---|---|
| `Texas_Salary_Prediction_Report.ipynb` | This project report |
| `Final_texas_salary_prediction.pkl` | Trained & tuned Random Forest model |

---
